# 01 — Data Exploration

Explore the sampled FOMC corpus: meeting counts, date coverage, section availability, and passage length distributions.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS = ROOT / 'outputs'

## 1. Sampled meetings

In [ ]:
meetings = pd.read_csv(PROCESSED / 'sampled_meetings.csv', parse_dates=['date'])
print(f'Total meetings: {len(meetings)}')
meetings.groupby('period')['date'].agg(['count', 'min', 'max'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for period, grp in meetings.groupby('period'):
    ax.scatter(grp['date'], [period]*len(grp), s=60, label=period)
ax.set_xlabel('Date')
ax.set_title('Sampled FOMC meetings by period')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 2. Passages

In [ ]:
passages = pd.read_csv(OUTPUTS / 'passages.csv')
print(f'Total passages: {len(passages)}')
passages.groupby('period')['passage_id'].count().rename('passages')

In [ ]:
passages['char_len'] = passages['text'].str.len()
passages['word_count'] = passages['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['char_len', 'word_count'], ['Character length', 'Word count']):
    for period, grp in passages.groupby('period'):
        ax.hist(grp[col], bins=30, alpha=0.6, label=period)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend()
plt.suptitle('Passage length distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Section coverage
passages.groupby(['period', 'section'])['passage_id'].count().rename('passages').unstack(fill_value=0)